In [2]:
# Discounted Value of Ordinary, Simple, Certain Annuities -- with optional deferral
#
# This program covers the payouts from an ordinary, simple, certain annuity,
# including deferred annuities whose first payout is delayed by m periods.
# It is assumed that compounding and periodic payouts happen at the same time.
#
# Inputs:
#   i = interest rate per period during the payout phase (must always be provided)
#   q = initial investment amount at time 0
#   n = number of payouts
#   r = payment to investor per period
#   m = deferral periods before payouts begin (default 0; set m = 0 for no deferral)
#   j = interest rate per period during deferral (leave at 0 to default to i)
#
# Leave EXACTLY ONE of n, q, or r at 0 for that variable to be computed.
#
# Formulas used:
#   Q = R * (1+j)^(-m) * [1 - (1+i)^(-n)] / i
#   R = i * Q * (1+j)^m / [1 - (1+i)^(-n)]
#   n = [ln R - ln(R - i * Q')] / ln(1+i),  where Q' = Q * (1+j)^m
#
# Setting m = 0 recovers the undeferred payout case.

import math
import tkinter as tk
from tkinter import messagebox
from functools import partial


# ---------------------------------------------------------------
# Pure calculation logic (importable and testable on its own)
# ---------------------------------------------------------------
def solve_annuity_payout(i, q=0.0, n=0.0, r=0.0, m=0.0, j=0.0):
    """
    Solve for the unknown variable in an ordinary, simple, certain
    annuity payout, with optional deferral.

    Returns (kind, value, display_text) where kind is one of
      'n', 'q', 'r', 'infinite', 'error'.
    """
    if i is None or i <= 0:
        return ("error", None, "Interest rate per period (i) must be > 0.")
    if m < 0:
        return ("error", None, "Deferral periods (m) must be >= 0.")
    if j < 0:
        return ("error", None, "Deferral-phase rate (j) must be >= 0.")

    j_eff = j if j > 0 else i  # default deferral rate = payout rate

    missing = sum(1 for v in (n, q, r) if v == 0)
    if missing != 1:
        return ("error", None,
                "Leave exactly one of n, q, or r at 0 "
                f"(currently {missing} are blank).")

    fwd = (1 + j_eff) ** m       # accumulates q to the start of payouts
    bwd = (1 + j_eff) ** (-m)    # discounts payouts back to time 0

    if n == 0:
        q_prime = q * fwd  # value of fund when first payout would occur

        if r - i * q_prime <= 0:
            return ("infinite", math.inf,
                    "infinite (payout <= interest earned each period)")

        if r > q_prime * (1 + i):
            return ("error", None,
                    "Not possible: requested payout exceeds the fund value at "
                    "the end of the first payout period (q'*(1+i) = "
                    f"{q_prime * (1 + i):,.2f}, where q' = q*(1+j)^m).")

        n_val = (math.log(r) - math.log(r - i * q_prime)) / math.log(1 + i)
        return ("n", n_val, f"{n_val:,.4f}")

    if q == 0:
        q_val = r * bwd * (1 - (1 + i) ** (-n)) / i
        return ("q", q_val, f"${q_val:,.2f}")

    # solve for r
    r_val = i * q * fwd / (1 - (1 + i) ** (-n))
    return ("r", r_val, f"${r_val:,.2f}")


# ---------------------------------------------------------------
# Tkinter GUI
# ---------------------------------------------------------------
root = tk.Tk()
root.title("Simple Annuity Payout Calculator (with optional deferral)")
root.lift()
root.attributes("-topmost", True)

# Holders for the result widgets so we can replace them on each compute
result_label = None
result_text = None


def on_quit():
    root.destroy()


def _destroy_result():
    """Remove any previously displayed result widgets."""
    global result_label, result_text
    for widget in (result_label, result_text):
        try:
            if widget is not None:
                widget.destroy()
        except Exception:
            pass
    result_label = None
    result_text = None


def _show_result(label_text, value_text):
    global result_label, result_text
    _destroy_result()
    result_label = tk.Label(root, text=label_text)
    result_label.grid(row=11, column=1, padx=3, pady=5, sticky="w")
    result_text = tk.Text(root, height=1, width=16)
    result_text.insert(tk.END, value_text)
    result_text.grid(row=11, column=2, padx=3, sticky="w")


def clear():
    _destroy_result()
    for entry, var in zip(
        (q1, q2, q3, q4, q5, q6),
        (w_q, x_i, y_n, z_r, m_var, j_var),
    ):
        entry.delete(0, tk.END)
        var.set(0)


def _safe_get(var):
    try:
        return var.get()
    except Exception:
        return 0


def compute(w_q, x_i, y_n, z_r, m_var, j_var):
    q = _safe_get(w_q)
    i = _safe_get(x_i)
    n = _safe_get(y_n)
    r = _safe_get(z_r)
    m = _safe_get(m_var)
    j = _safe_get(j_var)

    kind, _val, text = solve_annuity_payout(i=i, q=q, n=n, r=r, m=m, j=j)

    if kind == "error":
        messagebox.showerror("Input Error", text)
        return
    if kind == "infinite":
        _show_result("Number of payouts:", text)
        return

    label_map = {
        "n": "Number of payouts:",
        "q": "Initial investment:",
        "r": "Payout per period:",
    }
    _show_result(label_map[kind], text)


# ----- Input fields -----
tk.Label(root,
         text="Leave exactly one of n, q, or r at 0 to have it computed.",
         fg="grey").grid(row=0, column=1, columnspan=2,
                         padx=3, pady=(8, 2), sticky="w")
tk.Label(root, text="(Set m = 0 for no deferral.)", fg="grey").grid(
    row=0, column=3, padx=3, pady=(8, 2), sticky="w")

tk.Label(root, text="Initial investment in annuity (q):").grid(
    row=1, column=1, padx=3, sticky="w")
w_q = tk.DoubleVar()
q1 = tk.Entry(root, textvariable=w_q)
q1.grid(row=1, column=2, padx=5, pady=3, sticky="w")

tk.Label(root,
         text="Interest per period during payout - must always be provided (i):"
         ).grid(row=2, column=1, padx=3, sticky="w")
x_i = tk.DoubleVar()
q2 = tk.Entry(root, textvariable=x_i)
q2.grid(row=2, column=2, padx=5, pady=3, sticky="w")

tk.Label(root, text="Number of payouts (n):").grid(
    row=3, column=1, padx=3, sticky="w")
y_n = tk.DoubleVar()
q3 = tk.Entry(root, textvariable=y_n)
q3.grid(row=3, column=2, padx=5, pady=3, sticky="w")

tk.Label(root, text="Amount of each payout (r):").grid(
    row=4, column=1, padx=3, sticky="w")
z_r = tk.DoubleVar()
q4 = tk.Entry(root, textvariable=z_r)
q4.grid(row=4, column=2, padx=5, pady=3, sticky="w")

tk.Label(root, text="Deferral periods before payouts begin (m):").grid(
    row=5, column=1, padx=3, sticky="w")
m_var = tk.DoubleVar()
q5 = tk.Entry(root, textvariable=m_var)
q5.grid(row=5, column=2, padx=5, pady=3, sticky="w")

tk.Label(root,
         text="Interest rate during deferral - leave at 0 to use i (j):"
         ).grid(row=6, column=1, padx=3, sticky="w")
j_var = tk.DoubleVar()
q6 = tk.Entry(root, textvariable=j_var)
q6.grid(row=6, column=2, padx=5, pady=3, sticky="w")

# ----- Buttons -----
compute_bound = partial(compute, w_q, x_i, y_n, z_r, m_var, j_var)
tk.Button(root, text="Compute Remaining Variable",
          command=compute_bound).grid(row=8, column=1, padx=3, pady=7)

tk.Button(root, text="Clear", command=clear).grid(
    row=25, column=1, pady=5)
tk.Button(root, text="Quit", command=on_quit).grid(
    row=25, column=2, padx=3, pady=5, sticky="w")

root.mainloop()